<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Finding 1

The FlyRank research paper reports relationships between search/content signals and search performance.

**Methodology question:** How is the outcome label defined? I would check that the label represents an outcome that occurs after the information used as features, so that the model does not learn from future information.

### Finding 2

The research paper reports model results using a validation design.

**Methodology question:** Does the validation setup test generalization to genuinely unseen data? In particular, I would check whether related observations from the same client can appear in both training and testing data. A grouped or time-aware split can provide a more conservative test.

These are constructive methodology questions rather than claims that the findings are incorrect.## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My Week-5 model used a random train/test split. Because multiple content pages can belong to the same client, a random split may allow the model to learn patterns that are specific to clients appearing in both sets.

For a more honest generalization test, I use GroupShuffleSplit with client_hash_id as the grouping variable.

I compare the original random split with the grouped-by-client split using the same features, Random Forest model, and classification metrics.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'query_diversity'
]

model_data = data.dropna(subset=feature_cols).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

# -------------------------
# BEFORE: Random split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

print("===== BEFORE: RANDOM SPLIT =====")
print(classification_report(
    y_test,
    random_pred,
    digits=3
))


# -------------------------
# AFTER: Grouped by client
# -------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(X_test_group)

print("\n===== AFTER: GROUPED BY CLIENT =====")
print(classification_report(
    y_test_group,
    group_pred,
    digits=3
))

print("\nTraining clients:",
      groups.iloc[train_idx].nunique())

print("Testing clients:",
      groups.iloc[test_idx].nunique())

===== BEFORE: RANDOM SPLIT =====
              precision    recall  f1-score   support

           0      0.848     0.393     0.537      4122
           1      0.914     0.989     0.950     26945

    accuracy                          0.910     31067
   macro avg      0.881     0.691     0.743     31067
weighted avg      0.905     0.910     0.895     31067


===== AFTER: GROUPED BY CLIENT =====
              precision    recall  f1-score   support

           0      0.946     0.545     0.692      2237
           1      0.928     0.995     0.960     13154

    accuracy                          0.929     15391
   macro avg      0.937     0.770     0.826     15391
weighted avg      0.930     0.929     0.921     15391


Training clients: 39
Testing clients: 13


In [ ]:
import duckdb
import pandas as pd
import os
import getpass

# Get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

# Connect DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connected.")

DuckDB connected.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 50
    )

    SELECT *
    FROM windowed
""").df()

print("Features:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,302.0,2.0,11.738551
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,554.0,0.0,23.401005
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,543.0,0.0,28.054512
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,424.0,1.0,24.978010
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,1459.0,0.0,34.729708


In [ ]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count)
            AS visible_queries,

        ANY_VALUE(rare_impressions_share)
            AS rare_share,

        ANY_VALUE(anonymized_impressions_share)
            AS anon_share,

        MAX(impressions_90d)
            AS top_query_impressions,

        SUM(impressions_90d)
            AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = (
    qsignals['top_query_impressions']
    / qsignals['kept_impressions']
)

qsignals['query_diversity'] = (
    1 - qsignals['top_query_share']
)

data = features.merge(
    qsignals,
    on='content_hash_id',
    how='left'
)

# Create the Week-5 target
data['is_declining'] = (
    data['imp_last30'] <
    0.8 * data['imp_prev30']
).astype(int)

print("Final data:", len(data))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final data: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,query_diversity,is_declining
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,302.0,2.0,11.738551,5.0,0.132000,0.678000,26.0,95.0,0.273684,0.726316,1
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,554.0,0.0,23.401005,4.0,0.130841,0.562083,121.0,230.0,0.526087,0.473913,1
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,543.0,0.0,28.054512,1.0,0.169108,0.797603,25.0,25.0,1.000000,0.000000,1
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,424.0,1.0,24.978010,5.0,0.186380,0.620072,33.0,108.0,0.305556,0.694444,1
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,1459.0,0.0,34.729708,12.0,0.056039,0.817193,37.0,233.0,0.158798,0.841202,1


In [ ]:
import os
import getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Connected successfully")

Connected successfully


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY
            f.client_hash_id,
            f.content_hash_id

        HAVING imp_prev30 >= 50
    )

    SELECT *
    FROM windowed
""").df()

print("Features:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features: (155903, 6)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,1060.0,1.0,12.267505
1,client_e547b89c05043229,content_4f6ed7741dfd65e4,9.0,98.0,0.0,28.857143
2,client_e547b89c05043229,content_bab118937886d46a,110.0,359.0,0.0,20.211330
3,client_e547b89c05043229,content_7206e9a3ceefb37a,279.0,226.0,1.0,22.038723
4,client_e547b89c05043229,content_0587243c78e9468a,48.0,180.0,0.0,26.178571


In [ ]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count)
            AS visible_queries,

        ANY_VALUE(rare_impressions_share)
            AS rare_share,

        ANY_VALUE(anonymized_impressions_share)
            AS anon_share,

        MAX(impressions_90d)
            AS top_query_impressions,

        SUM(impressions_90d)
            AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = (
    qsignals['top_query_impressions']
    / qsignals['kept_impressions']
)

qsignals['query_diversity'] = (
    1 - qsignals['top_query_share']
)

data = features.merge(
    qsignals,
    on='content_hash_id',
    how='left'
)

print("Data shape:", data.shape)
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data shape: (155903, 13)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,query_diversity
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,1060.0,1.0,12.267505,5.0,0.038401,0.897335,34.0,82.0,0.414634,0.585366
1,client_e547b89c05043229,content_4f6ed7741dfd65e4,9.0,98.0,0.0,28.857143,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,client_e547b89c05043229,content_bab118937886d46a,110.0,359.0,0.0,20.211330,1.0,0.168443,0.528785,142.0,142.0,1.000000,0.000000
3,client_e547b89c05043229,content_7206e9a3ceefb37a,279.0,226.0,1.0,22.038723,1.0,0.219802,0.748515,16.0,16.0,1.000000,0.000000
4,client_e547b89c05043229,content_0587243c78e9468a,48.0,180.0,0.0,26.178571,1.0,0.118421,0.763158,27.0,27.0,1.000000,0.000000


I checked the final feature set for information that could only become available after the prediction decision.

The target is is_declining, which is based on the last-30-day impression outcome.

The feature imp_prev30 represents the previous 30-day period and is available before the outcome window.

The query-level features describe the query mix and are used as supporting signals.

I deliberately exclude imp_last30 because it is part of the outcome window used to create the label. Including it would leak information about the outcome into the model.

In [ ]:
print("===== LEAKAGE AUDIT =====")

print("\nTarget:")
print("is_declining")

print("\nFeatures used:")
for feature in feature_cols:
    print("-", feature)

print("\nPotential outcome variable:")
print("- imp_last30")

print("\nLeakage check:")
print(
    "imp_last30 included:",
    'imp_last30' in feature_cols
)

print(
    "is_declining included as feature:",
    'is_declining' in feature_cols
)

===== LEAKAGE AUDIT =====

Target:
is_declining

Features used:
- imp_prev30
- visible_queries
- rare_share
- anon_share
- top_query_share
- query_diversity

Potential outcome variable:
- imp_last30

Leakage check:
imp_last30 included: False
is_declining included as feature: False


In [ ]:
data['is_declining'] = (
    data['imp_last30'] < 0.8 * data['imp_prev30']
).astype(int)

print(data['is_declining'].value_counts())

is_declining
1    136778
0     19125
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'query_diversity'
]

model_data = data.dropna(subset=feature_cols).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

# BEFORE: Random split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)
random_pred = random_model.predict(X_test)

print("===== BEFORE: RANDOM SPLIT =====")
print(classification_report(y_test, random_pred, digits=3))


# AFTER: Grouped by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(X_test_group)

print("\n===== AFTER: GROUPED BY CLIENT =====")
print(classification_report(
    y_test_group,
    group_pred,
    digits=3
))

print("\nTraining clients:",
      groups.iloc[train_idx].nunique())

print("Testing clients:",
      groups.iloc[test_idx].nunique())

===== BEFORE: RANDOM SPLIT =====
              precision    recall  f1-score   support

           0      0.842     0.394     0.537      4122
           1      0.914     0.989     0.950     26945

    accuracy                          0.910     31067
   macro avg      0.878     0.692     0.744     31067
weighted avg      0.905     0.910     0.895     31067


===== AFTER: GROUPED BY CLIENT =====
              precision    recall  f1-score   support

           0      0.950     0.544     0.691      2237
           1      0.928     0.995     0.960     13154

    accuracy                          0.930     15391
   macro avg      0.939     0.769     0.826     15391
weighted avg      0.931     0.930     0.921     15391


Training clients: 39
Testing clients: 13


### Original claim

The model predicts which content pages will decline and identifies which pages need optimization.

### Safer claim

The model shows measured ability to distinguish pages with observed impression declines under the tested validation setup. Its output can provide directional decision-support for prioritizing pages for human review.

The model does not prove why a page declined, does not establish causality, and does not predict Google's ranking algorithm.

In [ ]:
print("Final claim:")
print(
    "The model provides directional decision-support "
    "for prioritizing pages for human review."
)

print("\nClaims not made:")
print("- No causal claim")
print("- No claim of predicting Google's algorithm")
print("- No claim that every recommended page needs optimization")

Final claim:
The model provides directional decision-support for prioritizing pages for human review.

Claims not made:
- No causal claim
- No claim of predicting Google's algorithm
- No claim that every recommended page needs optimization


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.